# 05 · Control PID dinámico alineado con el P&ID de A3110

Lazos (ver `docs/control_PID_P&ID.md`):

| Lazo | Tag P&ID (A3110) | PV | MV |
|---|---|---|---|
| A | `PIRCSAHL 3101` (vacío) | T del cristalizador | vapor extraído `m_evap` |
| B | `LIRCSAHL 3101` | volumen V | descarga `Q_out` |
| C | `OIRCAHL 3101 → FIRCAL 3101` | densidad de pulpa (maestro) → caudal de dilución (esclavo) | `Q_dil` |
| D | *propuesto, no está en el P&ID* | ppm ClO₄ (laboratorio, retardo 1 h, cada 2 h) | consigna de densidad del lazo C |

Las sintonías salen de ensayos de escalón (FOPDT + SIMC), no de números elegidos a mano. El pareamiento A/B/C es una hipótesis a confirmar con la leyenda del plano.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))           # raíz del repo (donde está msmpr_core)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from msmpr_core.params import make_params, placeholders, PROVENANCE
from msmpr_core.plant import Plant, simulate
from msmpr_core import control as C
os.makedirs("../docs/fig", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
p = make_params()
pl = Plant(p)
x0, u0, d0 = pl.steady_state(alpha=0.25)
m0 = pl.metrics(x0, u0, d0)
tun, info = C.autotune(pl, x0, u0, d0)
tp, ip, rho_lim = C.tune_ppm_loop(pl)
tun["ppm"] = tp
tabla = pd.DataFrame({k: {"Kp": v.Kp, "Ki [1/s]": v.Ki, "acción directa": v.direct} for k, v in tun.items()}).T
print("Modelos identificados:"); [print(" ", k, {a: (round(b, 4) if isinstance(b, (int, float, np.floating)) else b) for a, b in v.items()}) for k, v in info.items()]
print("Lazo D:", {a: round(float(b), 4) for a, b in ip.items()})
tabla

## Escenario de perturbaciones
* **t = 2 h:** la alimentación sube +6 °C y +12 % de caudal.
* **t = 6 h:** el ClO₄⁻ de alimentación sube ×1.5.

Se comparan: lazo abierto, lazo cerrado A+B+C y A+B+C+D.

In [ ]:
def dist(t):
    d = dict(d0)
    if t >= 2*3600: d["T_feed"] += 6.0; d["Q_stock"] *= 1.12
    if t >= 6*3600: d["CP_stock"] *= 1.5
    return d

sp = {"T": x0[1], "V": x0[0], "rho": m0["slurry_density_kg_m3"]}
T_END, DT = 14*3600, 10.0

ol = simulate(pl, x0, T_END, DT, lambda t, x, pl_: u0, disturbance=dist)
cl = simulate(pl, x0, T_END, DT, C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "rho")), disturbance=dist)
lab = C.SampledSensor(7200.0, 3600.0, m0["ppm_ClO4"])
ctrlD = C.PlantController(pl, u0, sp, tun, enabled=("T", "V", "rho", "ppm"), ppm_sp=m0["ppm_ClO4"], lab=lab)
ctrlD.pid_ppm.u_min, ctrlD.pid_ppm.u_max = sp["rho"] - rho_lim, sp["rho"] + rho_lim
clD = simulate(pl, x0, T_END, DT, ctrlD, disturbance=dist)

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
S = {"Lazo abierto": (ol, "tab:gray", "--"), "A+B+C": (cl, "tab:blue", "-"), "A+B+C+D (propuesto)": (clD, "teal", "-")}
for name, (df, col, ls) in S.items():
    ax[0,0].plot(df.t_h, df["T"], color=col, ls=ls, label=name)
    ax[0,1].plot(df.t_h, df.u_m_evap*3.6, color=col, ls=ls)
    ax[1,0].plot(df.t_h, df["V"], color=col, ls=ls)
    ax[1,1].plot(df.t_h, df.u_Q_out*3600, color=col, ls=ls)
    ax[2,0].plot(df.t_h, df.ppm_ClO4, color=col, ls=ls)
    ax[2,1].plot(df.t_h, df.u_Q_dil_cmd*3600, color=col, ls=ls)
ax[0,0].axhline(sp["T"], color="g", ls=":"); ax[1,0].axhline(sp["V"], color="g", ls=":")
ax[2,0].axhline(p["spec_ClO4_ppm_max"], color="r", ls="-", lw=1); ax[2,0].text(0.2, p["spec_ClO4_ppm_max"]+0.5, "spec provisional 40 ppm", color="r", fontsize=8)
for a in ax.flat:
    a.axvline(2, color="orange", ls="-.", alpha=.6); a.axvline(6, color="red", ls="-.", alpha=.6)
ax[0,0].set(ylabel="T [°C]", title="A · PV: temperatura (PIRCSAHL 3101)"); ax[0,0].legend(fontsize=8)
ax[0,1].set(ylabel="vapor [t/h]", title="A · MV: vapor extraído")
ax[1,0].set(ylabel="V [m³]", title="B · PV: volumen (LIRCSAHL 3101)"); ax[1,1].set(ylabel="m³/h", title="B · MV: descarga Q_out")
ax[2,0].set(ylabel="ClO₄⁻ producto [ppm]", xlabel="t [h]", title="Calidad (laboratorio simulado)"); ax[2,1].set(ylabel="m³/h", xlabel="t [h]", title="C · MV: agua de dilución (FIRCAL 3101)")
fig.suptitle("Respuesta de los lazos del P&ID de A3110 (parámetros placeholder)", fontweight="bold"); fig.tight_layout()
fig.savefig("../docs/fig/05_control_PID.png"); plt.show()

In [ ]:
def resumen(df, name):
    return {"caso": name, "T_min": df["T"].min(), "T_max": df["T"].max(), "V_min": df["V"].min(), "V_max": df["V"].max(),
            "ppm_max": df.ppm_ClO4.max(), "Y_final_%": df.recovery_pct.iloc[-1]}
pd.DataFrame([resumen(ol, "Lazo abierto"), resumen(cl, "A+B+C"), resumen(clD, "A+B+C+D")]).round(2)

## Lectura y límites
* **A y B:** sin control, la T sube a ≈45 °C y el volumen rebasa; con PID quedan dentro de décimas de °C y de unos m³. Es lo esperable para lazos bien sintonizados sobre este modelo.
* **C:** la densidad es una señal débil con la pulpa del caso base; con perturbación la dilución cae a su límite. Antes de confiar en este lazo hay que fijar la solubilidad y la densidad de pulpa reales.
* **D:** reduce el pico de ClO₄⁻ a costa de producción: es el conflicto del frente de Pareto. No lleva el ppm bajo 40 porque su autoridad (consigna de densidad ±) está acotada y el ClO₄ de alimentación y el `Kd` son supuestos.
* No se modelan: ruido de medición, tiempos muertos de transporte, saturación de válvulas reales, interacciones con las bombas de circulación (`SIC 3102`) ni el agitador (`SIC 3151`).